In [15]:
import pandas as pd
import numpy as np
import re
import nltk
import os
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to d:\kuliah sem
[nltk_data]     4\nlp\nlp_env\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to d:\kuliah sem
[nltk_data]     4\nlp\nlp_env\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to d:\kuliah sem
[nltk_data]     4\nlp\nlp_env\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Load GitHub Issues Dataset

Load from multiple sources with fallback:
1. Hugging Face downloaded CSV
2. Local processed CSV
3. Local raw CSV

In [ ]:
data_sources = [
    '../data/raw/github_issues_hf.csv',      # From HF download
    '../data/raw/github_issues_hf.csv',      # Raw local
]

df = None
for path in data_sources:
    if os.path.exists(path):
        print(f"Loading from: {path}")
        df = pd.read_csv(path)
        break

if df is None:
    print("ERROR: No dataset found!")
    print("Please run 01_load_huggingface_dataset.ipynb first")
    print("Or place a CSV file in data/raw/ or data/processed/")
    raise FileNotFoundError("Dataset not found")

print(f"\nDataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Loading from: ../data/raw/github_issues_hf.csv

Dataset Shape: (114073, 7)

Columns: ['id', 'repo', 'title', 'body', 'labels', 'priority', 'severity']

First few rows:


,id,repo,title,body,labels,priority,severity
0,393061,youtube-dl,Output file size with -s or -g,Was: http://bitbucket.org/rg3/youtube-dl/issue...,request,medium,Critical
1,1637737,youtube-dl,Create a php API and demo page,youtube-dl is often embedded by php applicatio...,php,low,Major
2,1639054,youtube-dl,"integrate template ""special sequences"" in help...",like in http://rg3.github.com/youtube-dl/docum...,request,low,Minor
3,1789251,youtube-dl,Add a path option to --keep-video,"Hey there,\n\nI think it would be a great idea...",request,low,Minor
4,1789512,youtube-dl,add support for picasaweb.google.com video clips,> /opt/local/bin/youtube-dl -t https://picasaw...,site-support-request,low,Minor


## Explore Dataset

Understand the structure and identify text and label columns.

In [17]:
# Display dataset info
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Show sample data
print(f"\n\nSample Data:")
for col in df.columns:
    print(f"\n{col}:")
    if df[col].dtype == 'object':
        print(f"  Sample: {str(df[col].iloc[0])[:150]}...")
    else:
        print(f"  Sample: {df[col].iloc[0]}")

Dataset Information:
Shape: (114073, 7)

Data Types:
id           int64
repo        object
title       object
body        object
labels      object
priority    object
severity    object
dtype: object

Missing Values:
id            0
repo          0
title         0
body        218
labels        0
priority      0
severity      0
dtype: int64

Duplicate Rows: 0

Memory Usage: 621.78 MB


Sample Data:

id:
  Sample: 393061

repo:
  Sample: youtube-dl...

title:
  Sample: Output file size with -s or -g...

body:
  Sample: Was: http://bitbucket.org/rg3/youtube-dl/issue/141/

If the file size was outputted it would be possible to script youtube-dl to test if the current v...

labels:
  Sample: request...

priority:
  Sample: medium...

severity:
  Sample: Critical...


## Identify Text and Label Columns

Automatically detect which columns contain text and labels.

In [18]:
# Identify text column (usually 'text', 'body', 'title', 'issue', 'description')
text_candidates = ['text', 'body', 'title', 'issue', 'description', 'content']
text_col = None
for col in text_candidates:
    if col in df.columns:
        text_col = col
        break

# Identify label column (usually 'severity', 'label', 'priority', 'importance')
label_candidates = ['severity', 'label', 'priority', 'importance', 'category']
label_col = None
for col in label_candidates:
    if col in df.columns:
        label_col = col
        break

print(f"Text Column: {text_col}")
print(f"Label Column: {label_col}")

if text_col is None or label_col is None:
    print("\nAvailable columns:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")
    print("\nPlease manually set text_col and label_col below")
else:
    print(f"\nColumns identified successfully!")
    print(f"Text samples ({text_col}):")
    print(df[text_col].iloc[0][:200])
    print(f"\nLabel distribution ({label_col}):")
    print(df[label_col].value_counts())

Text Column: body
Label Column: severity

Columns identified successfully!
Text samples (body):
Was: http://bitbucket.org/rg3/youtube-dl/issue/141/

If the file size was outputted it would be possible to script youtube-dl to test if the current video the the harddrive is in the best possible qua

Label distribution (severity):
severity
Critical    66491
Minor       29769
Major       17813
Name: count, dtype: int64


## Text Cleaning Function

Comprehensive text cleaning pipeline for GitHub issues:
- Lowercase
- Remove URLs
- Remove code snippets
- Remove stack traces
- Remove special characters (keep numbers & underscores)
- Tokenize
- Remove stopwords
- Lemmatization

In [19]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs (http://, https://, www)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 3. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 4. Remove code snippets (backticks)
    text = re.sub(r'`[^`]*`', '', text)
    
    # 5. Remove code blocks (triple backticks)
    text = re.sub(r'```[\s\S]*?```', '', text)
    
    # 6. Remove file paths and drive letters
    text = re.sub(r'([a-zA-Z]:\\[\\\S|*\S]?.*)', '', text)
    text = re.sub(r'/[\w/\-\.]+\.[a-z]+', '', text)
    
    # 7. Remove stack traces (lines starting with 'at' or containing line numbers)
    text = re.sub(r'at\s+\w+[\.\w]+\(.*?\)', '', text)
    text = re.sub(r'line\s+\d+', '', text, flags=re.IGNORECASE)
    
    # 8. Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # 9. Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # 10. Remove special characters (keep numbers & underscores for tech terms)
    text = re.sub(r'[^a-z0-9\s_]', '', text)
    
    # 11. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 12. Tokenize
    tokens = text.split()
    
    # 13. Remove stopwords and short tokens, then lemmatize
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    
    return ' '.join(tokens)

print("Text preprocessing function created!")

Text preprocessing function created!


## Test Preprocessing Function

See before and after examples.

In [20]:
# Test on sample data
print("PREPROCESSING EXAMPLES:")
print("=" * 80)

for idx in range(min(3, len(df))):
    original = df[text_col].iloc[idx]
    cleaned = preprocess_text(original)
    
    print(f"\nExample {idx + 1}:")
    print(f"ORIGINAL ({len(original)} chars):")
    print(original[:200] if len(original) > 200 else original)
    print(f"\nCLEANED ({len(cleaned)} chars):")
    print(cleaned[:200] if len(cleaned) > 200 else cleaned)
    print("-" * 80)

PREPROCESSING EXAMPLES:

Example 1:
ORIGINAL (394 chars):
Was: http://bitbucket.org/rg3/youtube-dl/issue/141/

If the file size was outputted it would be possible to script youtube-dl to test if the current video the the harddrive is in the best possible qua

CLEANED (228 chars):
file size outputted would possible script youtubedl test current video harddrive best possible quality happens youtube reencode video higher resolution able extract file size youtubedl would useful fi
--------------------------------------------------------------------------------

Example 2:
ORIGINAL (472 chars):
youtube-dl is often embedded by php applications. There should be an example php page included, with the following features:
- Enter URLs and execute youtube-dl (in the background)
- Show progress of 

CLEANED (349 chars):
youtubedl often embedded php application example php page included following feature enter url execute youtubedl background show progress youtubedl instance offer way abort youtube

## Apply Preprocessing to Full Dataset

This may take a few minutes for large datasets.

In [21]:
# Apply preprocessing to all text
print("Applying preprocessing to all rows...")
df['clean_text'] = df[text_col].apply(preprocess_text)

print(f"Done!")
print(f"\nOriginal text column: {text_col}")
print(f"New clean text column: clean_text")
print(f"\nDataset shape: {df.shape}")

Applying preprocessing to all rows...
Done!

Original text column: body
New clean text column: clean_text

Dataset shape: (114073, 8)


## Analyze Cleaned Data Quality

Check for empty texts and data quality.

In [22]:
# Check for empty or very short texts
empty_texts = (df['clean_text'].str.len() == 0).sum()
short_texts = (df['clean_text'].str.len() < 10).sum()

print("Data Quality Analysis:")
print(f"Total rows: {len(df)}")
print(f"Empty texts: {empty_texts} ({empty_texts/len(df)*100:.2f}%)")
print(f"Short texts (<10 chars): {short_texts} ({short_texts/len(df)*100:.2f}%)")
print(f"\nText length statistics:")
print(f"  Mean: {df['clean_text'].str.len().mean():.1f} characters")
print(f"  Median: {df['clean_text'].str.len().median():.1f} characters")
print(f"  Min: {df['clean_text'].str.len().min():.1f} characters")
print(f"  Max: {df['clean_text'].str.len().max():.1f} characters")

print(f"\nLabel distribution:")
print(df[label_col].value_counts())
print(f"\nLabel percentages:")
print(df[label_col].value_counts(normalize=True) * 100)

Data Quality Analysis:
Total rows: 114073
Empty texts: 583 (0.51%)
Short texts (<10 chars): 901 (0.79%)

Text length statistics:
  Mean: 623.3 characters
  Median: 422.0 characters
  Min: 0.0 characters
  Max: 113874.0 characters

Label distribution:
severity
Critical    66491
Minor       29769
Major       17813
Name: count, dtype: int64

Label percentages:
severity
Critical    58.288114
Minor       26.096447
Major       15.615439
Name: proportion, dtype: float64


## Remove Low Quality Data

Filter out empty and very short texts.

In [23]:
# Remove rows with empty or very short cleaned text
min_length = 10  # Minimum character length

print(f"Before filtering: {len(df)} rows")

df_filtered = df[df['clean_text'].str.len() >= min_length].copy()

print(f"After filtering (min_length={min_length}): {len(df_filtered)} rows")
print(f"Removed: {len(df) - len(df_filtered)} rows ({(len(df) - len(df_filtered))/len(df)*100:.2f}%)")

# Update df to filtered version
df = df_filtered.reset_index(drop=True)

print(f"\nNew label distribution:")
print(df[label_col].value_counts())

Before filtering: 114073 rows
After filtering (min_length=10): 113172 rows
Removed: 901 rows (0.79%)

New label distribution:
severity
Critical    66218
Minor       29267
Major       17687
Name: count, dtype: int64


## Handle Duplicates

Remove duplicate texts.

In [24]:
# Check for duplicates
duplicates_total = df.duplicated().sum()
duplicates_text = df['clean_text'].duplicated().sum()

print(f"Total duplicate rows: {duplicates_total}")
print(f"Duplicate text entries: {duplicates_text}")

# Remove complete duplicates
df = df.drop_duplicates(subset=['clean_text'], keep='first').reset_index(drop=True)

print(f"\nAfter removing duplicate texts: {len(df)} rows")
print(f"\nLabel distribution:")
print(df[label_col].value_counts())

Total duplicate rows: 0
Duplicate text entries: 908

After removing duplicate texts: 112264 rows

Label distribution:
severity
Critical    65527
Minor       29079
Major       17658
Name: count, dtype: int64


## Prepare Final Preprocessed Dataset

Select only clean_text and label columns.

In [25]:
# Create final dataset with only necessary columns
df_preprocessed = df[['clean_text', label_col]].copy()
df_preprocessed.columns = ['text', 'severity']  # Standardize column names

print("Final Preprocessed Dataset:")
print(f"Shape: {df_preprocessed.shape}")
print(f"\nColumns: {df_preprocessed.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_preprocessed.head(10))
print(f"\nLabel distribution:")
print(df_preprocessed['severity'].value_counts())
print(f"\nData types:")
print(df_preprocessed.dtypes)

Final Preprocessed Dataset:
Shape: (112264, 2)

Columns: ['text', 'severity']

First few rows:
                                                text  severity
0  file size outputted would possible script yout...  Critical
1  youtubedl often embedded php application examp...     Major
2  hey think would great idea ability save video ...     Minor
3  optlocalbinyoutubedl warning falling back gene...     Minor
4  updated descrption trait object trait object h...  Critical
5  following patch send encoding progress output ...  Critical
6  function reimplemented per architecture doc on...  Critical
7  would wonderful option download video given st...     Major
8  possible support downloads course assuming cou...     Major
9  hello default extractaudio era video file next...     Minor

Label distribution:
severity
Critical    65527
Minor       29079
Major       17658
Name: count, dtype: int64

Data types:
text        object
severity    object
dtype: object


## Save Preprocessed Data

Save cleaned data for feature extraction.

In [26]:
# Menggabungkan Major dan Minor menjadi Non-Critical
df['severity'] = df['severity'].replace(['Major', 'Minor'], 'Non-Critical')

# Cek distribusi baru
print("Distribusi Label Baru:")
print(df['severity'].value_counts())

Distribusi Label Baru:
severity
Critical        65527
Non-Critical    46737
Name: count, dtype: int64


In [27]:
# Create output directory if needed
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# Save preprocessed data
output_path = os.path.join(output_dir, 'github_issues_preprocessed.csv')
df_preprocessed.to_csv(output_path, index=False)

print(f"Preprocessed data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Also save as parquet for faster loading
parquet_path = os.path.join(output_dir, 'github_issues_preprocessed.parquet')
df_preprocessed.to_parquet(parquet_path, index=False)
print(f"Parquet file saved to: {parquet_path}")
print(f"File size: {os.path.getsize(parquet_path) / 1024 / 1024:.2f} MB")

Preprocessed data saved to: ../data/processed\github_issues_preprocessed.csv
File size: 68.56 MB
Parquet file saved to: ../data/processed\github_issues_preprocessed.parquet
File size: 33.24 MB


## Preprocessing Summary

Overview of preprocessing steps and results.

In [28]:
print("="*80)
print("PREPROCESSING COMPLETE")
print("="*80)
print(f"\nSteps performed:")
print(f"  1. Loaded dataset from: {[s for s in data_sources if os.path.exists(s)][0]}")
print(f"  2. Lowercased text")
print(f"  3. Removed URLs, emails, and code snippets")
print(f"  4. Removed stack traces and file paths")
print(f"  5. Removed HTML tags and special characters")
print(f"  6. Tokenized text")
print(f"  7. Removed stopwords")
print(f"  8. Applied lemmatization")
print(f"  9. Removed empty and short texts")
print(f" 10. Removed duplicate texts")
print(f"\nResults:")
print(f"  - Output shape: {df_preprocessed.shape}")
print(f"  - Saved to: {output_path}")
print(f"  - Format: CSV and Parquet")
print(f"\nLabel distribution:")
for label, count in df_preprocessed['severity'].value_counts().items():
    pct = count / len(df_preprocessed) * 100
    print(f"  {label:10s}: {count:6d} ({pct:5.2f}%)")
print(f"\nNext step: Run 03_feature_extraction.ipynb")
print("="*80)

PREPROCESSING COMPLETE

Steps performed:
  1. Loaded dataset from: ../data/raw/github_issues_hf.csv
  2. Lowercased text
  3. Removed URLs, emails, and code snippets
  4. Removed stack traces and file paths
  5. Removed HTML tags and special characters
  6. Tokenized text
  7. Removed stopwords
  8. Applied lemmatization
  9. Removed empty and short texts
 10. Removed duplicate texts

Results:
  - Output shape: (112264, 2)
  - Saved to: ../data/processed\github_issues_preprocessed.csv
  - Format: CSV and Parquet

Label distribution:
  Critical  :  65527 (58.37%)
  Minor     :  29079 (25.90%)
  Major     :  17658 (15.73%)

Next step: Run 03_feature_extraction.ipynb
